In [ ]:
"""
Merge multiple survival analysis datasets (Parquet files) with ULTRA-MINIMAL RAM.

This script combines multiple labeled dataset Parquet files into a single merged
dataset by streaming them in small chunks (batches). It completely avoids loading
even a single full file into memory at once.

Configuration:
    PARQUET_FILES: List of Parquet file paths to merge.
    OUTPUT_PATH: Output path for merged dataset.
    BATCH_SIZE: Number of rows to process at a time (lower = less RAM).
    VERBOSE: Print progress messages.
"""

from __future__ import annotations

import sys
import gc
from pathlib import Path
from typing import Optional
from collections import Counter

import pyarrow as pa
import pyarrow.parquet as pq
from enum import IntEnum


class EventType(IntEnum):
    CENSORED = 0
    FAVORABLE_FILL = 1
    TOXIC_FILL = 2

    @classmethod
    def is_valid(cls, value: int) -> bool:
        return value in cls._value2member_map_


# ============================================================================
# USER CONFIGURATION
# ============================================================================

# List of Parquet files to merge
PARQUET_FILES: list[str] = [
    "/content/drive/MyDrive/11-785 Project/data/datasets/dataset_XNAS_ITCH_AAPL_mbo_20251201_20260101_3modes.parquet",
    "/content/drive/MyDrive/11-785 Project/data/datasets/dataset_XNAS_ITCH_AAPL_mbo_20251101_20251201_3modes.parquet",
    "/content/drive/MyDrive/11-785 Project/data/datasets/dataset_XNAS_ITCH_AAPL_mbo_20251001_20251101_3modes.parquet"
]

# Output path for merged dataset
OUTPUT_PATH: str | None = "/content/drive/MyDrive/11-785 Project/data/datasets/merged_dataset_XNAS_ITCH_AAPL_mbo_20251001_20260101_3modes.parquet"

# Number of rows to load into RAM at a single time
# 100,000 is usually very safe. If you still OOM, drop this to 50,000 or 10,000.
BATCH_SIZE: int = 100_000

# Print progress messages
VERBOSE: bool = True


def print_merge_statistics(
    dfs_info: list[dict],
    total_rows: int,
    columns: list[str],
    event_counts: Counter,
    prefix: str = "",
) -> None:
    """Print detailed statistics about the merged datasets."""
    print(f"\n{prefix}{'=' * 70}")
    print(f"{prefix}MERGE STATISTICS (CHUNKED STREAMING)")
    print(f"{prefix}{'=' * 70}")

    # Input datasets
    print(f"\n{prefix}Input Datasets ({len(dfs_info)}):")
    for info in dfs_info:
        print(f"{prefix}  - {info['name']}: {info['rows']:,} rows")

    # Merged dataset
    print(f"\n{prefix}Merged Dataset:")
    print(f"{prefix}  - Total rows: {total_rows:,}")
    print(f"{prefix}  - Columns: {len(columns)}")
    print(f"{prefix}  - Column names: {columns}")

    # Event statistics
    if event_counts:
        print(f"\n{prefix}Event Type Distribution:")
        for event_code, count in sorted(event_counts.items()):
            try:
                event_name = EventType(event_code).name
            except ValueError:
                event_name = f"UNKNOWN({event_code})"
            pct = 100 * count / total_rows if total_rows > 0 else 0
            print(f"{prefix}  - {event_name}: {count:,} ({pct:.2f}%)")

    print(f"\n{prefix}{'=' * 70}\n")


def merge_parquet_datasets_chunked(
    parquet_files: list[Path],
    output_path: Path,
    batch_size: int = 100_000,
    verbose: bool = True,
) -> None:
    """
    Merge multiple Parquet files by streaming them in small chunks.
    This uses O(BATCH_SIZE) RAM, making it nearly immune to OOM errors.
    """
    if not parquet_files:
        raise ValueError("No Parquet files provided to merge.")
    if not output_path:
        raise ValueError("OUTPUT_PATH is required for chunked streaming.")

    # Validate all files exist
    for file in parquet_files:
        if not file.exists():
            raise FileNotFoundError(f"Parquet file not found: {file}")

    if verbose:
        print(f"Streaming and merging {len(parquet_files)} dataset(s) in chunks of {batch_size:,} rows...")

    output_path.parent.mkdir(parents=True, exist_ok=True)

    writer = None
    dfs_info = []
    total_rows = 0
    event_counts = Counter()
    merged_columns = []

    try:
        for i, file in enumerate(parquet_files, 1):
            if verbose:
                print(f"[{i}/{len(parquet_files)}] Processing {file.name}...")
            
            # Open the parquet file for reading (does NOT load it into RAM)
            parquet_file = pq.ParquetFile(file)
            
            if not merged_columns:
                merged_columns = parquet_file.schema.names
            
            file_rows = 0
            
            # Iterate through the file in small batches
            for batch in parquet_file.iter_batches(batch_size=batch_size):
                
                # 1. Initialize writer with the schema of the first batch
                if writer is None:
                    writer = pq.ParquetWriter(output_path, batch.schema)
                
                # 2. Write the small chunk directly to the output file
                writer.write_batch(batch)
                
                # 3. Update statistics
                batch_rows = batch.num_rows
                file_rows += batch_rows
                total_rows += batch_rows
                
                if "event_type" in merged_columns:
                    # Extract just the event_type column to count it
                    events = batch.column("event_type").to_pylist()
                    event_counts.update(events)
                
                # Force garbage collection on the batch
                del batch
                gc.collect()
                
            dfs_info.append({"name": file.name, "rows": file_rows})
            
            if verbose:
                print(f"    -> Finished {file.name}: {file_rows:,} rows processed.")

    finally:
        # Ensure the file stream is safely closed
        if writer is not None:
            writer.close()

    if verbose:
        print(f"\n✓ Merged dataset successfully saved to {output_path}")
        print_merge_statistics(
            dfs_info=dfs_info,
            total_rows=total_rows,
            columns=merged_columns,
            event_counts=event_counts
        )


def main():
    """Main entry point."""
    try:
        if not PARQUET_FILES:
            raise ValueError("PARQUET_FILES is empty. Set the files to merge.")
        if not OUTPUT_PATH:
            raise ValueError("OUTPUT_PATH must be set.")

        parquet_paths = [Path(f) for f in PARQUET_FILES]
        output_path = Path(OUTPUT_PATH)

        merge_parquet_datasets_chunked(
            parquet_files=parquet_paths,
            output_path=output_path,
            batch_size=BATCH_SIZE,
            verbose=VERBOSE,
        )
    except Exception as e:
        print(f"Error: {e}", file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    main()